In [11]:
# ✅ CELL 1 (UPDATED) — support/setup with template-agnostic text extraction
# Goal: work across ANY template without hardcoding per-template logic.
# We just recursively collect all string fields (light filtering), then score candidates.

import os, json, re, time, hashlib, random, asyncio
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter
from difflib import SequenceMatcher
from dotenv import load_dotenv
from openai import AsyncAzureOpenAI

# -----------------------------
# PATHS (repo-standard)
# -----------------------------
BASE = Path("../03_Outputs")
EN_LESSONS_DIR = BASE / "SEA_Modules/en"
POLL_REGISTRY_PATH = BASE / "polls/poll_registry.json"
BACKUP_DIR = BASE / "polls/backups"

BACKUP_DIR.mkdir(parents=True, exist_ok=True)
POLL_REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)

if not EN_LESSONS_DIR.exists():
    raise FileNotFoundError(f"EN lessons dir not found: {EN_LESSONS_DIR.resolve()}")

# -----------------------------
# Azure OpenAI
# -----------------------------
load_dotenv()
AZURE_OPENAI_ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION= os.getenv("AZURE_OPENAI_API_VERSION")
if not (AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY and AZURE_OPENAI_DEPLOYMENT and AZURE_OPENAI_API_VERSION):
    raise ValueError("Missing Azure OpenAI env vars. Need AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_DEPLOYMENT, AZURE_OPENAI_API_VERSION")

client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# -----------------------------
# Minimal helpers
# -----------------------------
def load_json(p: Path) -> Optional[dict]:
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return None

def save_json(obj: Any, p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def strip_html(s: str) -> str:
    return re.sub(r"<[^>]+>", "", s or "")

def sha256(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def short_hash(h: str, n: int = 6) -> str:
    return (h or "")[:n]

def extract_first_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                chunk = text[start:i+1]
                try:
                    return json.loads(chunk)
                except Exception:
                    return None
    return None

def poll_fingerprint(title: str, desc: str, options: List[str]) -> str:
    raw = json.dumps({
        "t": normalize_ws(strip_html(title)).lower(),
        "d": normalize_ws(strip_html(desc)).lower(),
        "o": [normalize_ws(strip_html(o)).lower() for o in options],
    }, ensure_ascii=False, sort_keys=True)
    return sha256(raw)

# -----------------------------
# Registry load / backup / indexes
# -----------------------------
def load_registry(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        save_json([], path)
        return []
    data = load_json(path)
    if data is None:
        return []
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and "polls" in data and isinstance(data["polls"], list):
        return data["polls"]
    raise ValueError("poll_registry.json must be a list OR {polls:[...]}")

registry: List[Dict[str, Any]] = load_registry(POLL_REGISTRY_PATH)

ts = time.strftime("%Y%m%d_%H%M%S")
backup_path = BACKUP_DIR / f"poll_registry_{ts}.json"
save_json(registry, backup_path)
print("✅ Backup saved:", backup_path)

def registry_indexes(reg: List[Dict[str, Any]]):
    per_lesson = {}
    exact_hashes = set()
    for r in reg:
        lid, pid = r.get("lesson_id"), r.get("pollId")
        if lid and pid:
            per_lesson.setdefault(lid, []).append(pid)
        ex = (r.get("fingerprints", {}) or {}).get("exact_hash")
        if ex:
            exact_hashes.add(ex)
    return per_lesson, exact_hashes

per_lesson, exact_hashes = registry_indexes(registry)

def build_avoid_list(reg: List[Dict[str, Any]], k: int = 20) -> List[str]:
    opts = []
    for r in reg[-800:]:
        c = (r.get("segment", {}) or {}).get("content", {}) or {}
        for o in (c.get("options", []) or []):
            if isinstance(o, dict):
                v = normalize_ws(strip_html(str(o.get("value","")))).lower()
                if v:
                    opts.append(v)
    if not opts:
        return []
    counts = Counter(opts)
    return [o for o,_ in counts.most_common(k)]

avoid_list = build_avoid_list(registry, k=20)

# -----------------------------
# Template-agnostic text extraction (scalable)
# -----------------------------
ALLOWED_STRING_KEYS = {
    # keep it light — common places where human-facing text lives
    "title","subtitle","label","intro","text","body","description","cta","prompt","quote","author","value","caption","footnote"
}
IGNORE_KEYS = {"src","url","href","imageRef","id","pollId","videoId","poster","path","filename","file","token","sas","key"}

def collect_strings(obj: Any, path: List[str]=None, out: List[str]=None) -> List[str]:
    # Recursively collect text-like strings from any template without special casing.
    if path is None: path = []
    if out is None: out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in IGNORE_KEYS:
                continue
            if isinstance(v, str):
                # only keep "likely UI text" fields OR long-ish free text
                if k in ALLOWED_STRING_KEYS or len(v.strip()) >= 40:
                    s = normalize_ws(strip_html(v))
                    if s:
                        out.append(s)
            else:
                collect_strings(v, path + [str(k)], out)
    elif isinstance(obj, list):
        for i, it in enumerate(obj):
            collect_strings(it, path + [str(i)], out)
    return out

def segment_text(seg: Dict[str, Any], max_chars: int = 2400) -> str:
    strings = collect_strings(seg)
    # de-dup while preserving order (simple)
    seen = set()
    uniq = []
    for s in strings:
        sl = s.lower()
        if sl in seen:
            continue
        seen.add(sl)
        uniq.append(s)
    txt = normalize_ws(" ".join(uniq))
    if len(txt) > max_chars:
        txt = txt[-max_chars:]  # keep tail; good for "poll after this segment"
    return txt

def segment_text_hash(seg: Dict[str, Any], mode: str, n_chars: int = 200) -> str:
    txt = segment_text(seg, max_chars=2200)
    if not txt:
        return ""
    snippet = txt[-n_chars:] if mode == "tail" else txt[:n_chars]
    return sha256(snippet)

def pick_candidates(segments: List[dict], top_k: int = 5) -> List[Dict[str, Any]]:
    cands = []
    for i in range(len(segments) - 1):
        b, a = segments[i], segments[i+1]
        if not isinstance(b, dict) or not isinstance(a, dict):
            continue

        a_tid = a.get("template_id","")
        if a_tid == "lesson_cover":
            continue

        b_txt = segment_text(b)
        plain = strip_html(b_txt)
        if len(plain) < 110:
            continue

        # minimal heuristic scoring (template-agnostic)
        score = 0
        if "?" in plain: score += 2
        if b.get("template_id") in {"echarts_chart","infographic","kpi_highlight_large","kpi_highlight_medium"}: score += 1
        if a.get("template_id") in {"lesson_part_cover","lesson_subpart_cover"}: score += 1

        cands.append({
            "score": score,
            "insert_index": i+1,
            "before": {"template_id": b.get("template_id",""), "text_hash": segment_text_hash(b, "tail")},
            "after":  {"template_id": a.get("template_id",""), "text_hash": segment_text_hash(a, "head")},
            "snippet": plain[-1100:]
        })

    cands.sort(key=lambda x: x["score"], reverse=True)
    return cands[:top_k]

def scan_en_lessons() -> List[str]:
    out=[]
    for p in sorted(EN_LESSONS_DIR.rglob("*.json")):
        if p.name == "module_structure.json":
            continue
        if re.match(r"^\d+\.\d+\.\d+\.json$", p.name):
            out.append(p.stem)
    return out

lesson_ids = scan_en_lessons()
print(f"📚 Found {len(lesson_ids)} EN lessons in {EN_LESSONS_DIR}")


✅ Backup saved: ../03_Outputs/polls/backups/poll_registry_20260130_151023.json
📚 Found 158 EN lessons in ../03_Outputs/SEA_Modules/en


In [12]:
# ✅ Cell 2 — simplest possible LLM poll generation
# - Generates up to 1 poll per lesson (skips lessons that already have >=1 poll in registry)
# - Minimal guardrails: not a quiz, avoid boring phrasing, avoid repeating exact option strings from avoid_list
# - Still stores anchors for robust reinsertion later.

TARGET_PER_LESSON = 1
MAX_ATTEMPTS = 4
TEMPERATURE = 0.9

def build_poll_segment(poll_id: str, title: str, desc: str, options: List[str]) -> Dict[str, Any]:
    return {
        "template_id": "poll",
        "color_scheme": "light",
        "content": {
            "pollId": poll_id,
            "title": title,
            "description": desc,
            "options": [{"id": i+1, "value": options[i]} for i in range(len(options))],
            "labels": {"submit":"Submit","cancel":"Cancel","edit":"Edit your response","votingAs":"Voting as"}
        }
    }

def is_boring_text(s: str) -> bool:
    s = (s or "").lower()
    return any(x in s for x in [
        "main factor","most influences","primary driver","key barrier","which factor","main barrier","primary challenge"
    ])

async def gen_one_poll(snippet: str) -> dict:
    prompt = f"""
Create ONE engaging interactive poll based on the lesson text below.

Rules:
- Not a quiz. No correct answer.
- Make it interesting + human. Something learners would actually enjoy answering.
- 3–5 options. All plausible. Options should be distinct (not synonyms).
- Avoid boring phrasing like "main factor", "most influences", "primary driver".
- Avoid reusing these exact option phrases if possible: {avoid_list}

Return ONLY valid JSON with this shape:
{{
  "title": "...",
  "description": "...?",
  "options": ["...", "...", "..."]
}}

Lesson text:
\"\"\"{snippet}\"\"\"
""".strip()

    resp = await client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role":"system","content":"Return ONLY strict JSON. Make the poll engaging and non-prescriptive."},
            {"role":"user","content":prompt},
        ],
        temperature=TEMPERATURE,
        max_tokens=500,
    )
    obj = extract_first_json_object((resp.choices[0].message.content or "").strip())
    if obj is None:
        raise ValueError("No parseable JSON returned")
    return obj

created = 0
failed = 0
skipped = 0

async def run_simple_generation():
    global registry, per_lesson, exact_hashes, avoid_list, created, failed, skipped

    for lid in lesson_ids:
        if len(per_lesson.get(lid, [])) >= TARGET_PER_LESSON:
            skipped += 1
            continue

        lesson_path = EN_LESSONS_DIR / f"Module_{lid.split('.')[0]}" / f"{lid}.json"
        lesson = load_json(lesson_path)
        if not lesson or not isinstance(lesson.get("segments"), list):
            failed += 1
            continue

        cands = pick_candidates(lesson["segments"], top_k=5)
        if not cands:
            failed += 1
            continue

        cand = cands[0]  # simplest: best candidate only
        snippet = cand["snippet"]

        last_err = None
        for _ in range(MAX_ATTEMPTS):
            try:
                out = await gen_one_poll(snippet)
                title = normalize_ws(out.get("title",""))
                desc  = normalize_ws(out.get("description",""))
                options = out.get("options", [])

                if not title or not desc or not isinstance(options, list) or not (3 <= len(options) <= 5):
                    last_err = "missing fields or wrong option count"
                    continue
                if is_boring_text(title + " " + desc):
                    last_err = "boring phrasing"
                    continue

                options = [normalize_ws(str(o)) for o in options if str(o).strip()]
                if len(options) < 3 or len(set([o.lower() for o in options])) != len(options):
                    last_err = "duplicate/empty options"
                    continue

                exact = poll_fingerprint(title, desc, options)
                if exact in exact_hashes:
                    last_err = "exact duplicate"
                    continue

                anchor_key = f"{cand['before']['template_id']}_to_{cand['after']['template_id']}"
                poll_id = f"poll-{lid}-{anchor_key}-{short_hash(exact, 6)}"

                record = {
                    "pollId": poll_id,
                    "lesson_id": lid,
                    "insert": {
                        "before": cand["before"],
                        "after":  cand["after"],
                    },
                    "segment": build_poll_segment(poll_id, title, desc, options),
                    "fingerprints": {"exact_hash": exact},
                    "meta": {
                        "created_at": time.strftime("%Y-%m-%d"),
                        "generator_version": "simple_prompt_v1",
                        "candidate_score": cand["score"],
                    }
                }

                registry.append(record)
                save_json(registry, POLL_REGISTRY_PATH)

                per_lesson.setdefault(lid, []).append(poll_id)
                exact_hashes.add(exact)
                avoid_list[:] = build_avoid_list(registry, k=20)  # keep avoid list fresh

                created += 1
                print(f"✅ {lid}: {poll_id}")
                break

            except Exception as e:
                last_err = str(e)

        else:
            failed += 1
            print(f"❌ {lid}: failed ({last_err})")

    print("\n📌 Simple Poll Generation Summary")
    print(f"✅ Created: {created}")
    print(f"⏭️ Skipped (already had poll): {skipped}")
    print(f"❌ Failed: {failed}")
    print(f"🗂 Registry: {POLL_REGISTRY_PATH.resolve()}")

await run_simple_generation()


✅ 1.0.0: poll-1.0.0-poll_to_text-2a8ecd
✅ 1.1.0: poll-1.1.0-list_of_lessons_to_poll-f88f42
✅ 1.1.1: poll-1.1.1-text_to_lesson_part_cover-cfdef1
✅ 1.1.2: poll-1.1.2-echarts_chart_to_poll-1f2baa
✅ 1.1.3: poll-1.1.3-echarts_chart_to_poll-a04a1d
✅ 1.2.0: poll-1.2.0-list_of_lessons_to_poll-785cb7
✅ 1.2.1: poll-1.2.1-poll_to_text-02f46d
✅ 1.2.2: poll-1.2.2-poll_to_text-94bbc7
✅ 1.2.3: poll-1.2.3-poll_to_text-420423
✅ 1.3.0: poll-1.3.0-list_of_lessons_to_poll-c863f5
✅ 1.3.1: poll-1.3.1-poll_to_text-fd60b7
✅ 1.3.2: poll-1.3.2-poll_to_lesson_subpart_cover-084baa
✅ 1.3.3: poll-1.3.3-poll_to_text-a1bb86
✅ 1.3.4: poll-1.3.4-poll_to_text-1677e0
❌ 1.3.5: failed (boring phrasing)
✅ 1.4.0: poll-1.4.0-list_of_lessons_to_poll-0534ad
✅ 1.4.1: poll-1.4.1-text_to_infographic-6d6011
✅ 1.4.2: poll-1.4.2-poll_to_lesson_subpart_cover-5edf29
✅ 1.4.3: poll-1.4.3-poll_to_lesson_part_cover-8cb945
✅ 2.0.0: poll-2.0.0-poll_to_text-71a198
✅ 2.1.0: poll-2.1.0-list_of_lessons_to_poll-f838b3
✅ 2.1.1: poll-2.1.1-poll_to_

In [6]:
# JUPYTER ONE-CELL: Insert polls from registry using anchor matching (safe for index shifts)
import json, re, hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

EN_BASE_FOLDER = Path("../03_Outputs/SEA_Modules/en")
POLL_REGISTRY_PATH = Path("../03_Outputs/Polls/poll_registry.json")

def lesson_path_en(lesson_id: str) -> Path:
    parts = lesson_id.split(".")
    if len(parts) != 3:
        raise ValueError(f"Invalid lesson_id: {lesson_id}")
    return EN_BASE_FOLDER / f"Module_{parts[0]}" / f"{lesson_id}.json"

def load_json(path: Path) -> Optional[dict]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def strip_html(s: str) -> str:
    return re.sub(r"<[^>]+>", "", s or "")

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def sha256(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def extract_text_from_segment(seg: Dict[str, Any]) -> str:
    if not isinstance(seg, dict):
        return ""
    tid = seg.get("template_id")
    ce = seg.get("content", {})
    parts = []

    if tid == "text":
        els = ce.get("text_elements", [])
        if isinstance(els, list):
            for el in els:
                if isinstance(el, dict):
                    c = el.get("content")
                    if isinstance(c, str) and c.strip():
                        parts.append(c)

    if tid == "key_concepts":
        kc = ce.get("key_concepts", [])
        if isinstance(kc, list):
            for item in kc:
                if isinstance(item, dict):
                    for k in ("title", "body"):
                        v = item.get(k)
                        if isinstance(v, str) and v.strip():
                            parts.append(v)

    if isinstance(ce, dict):
        for k in ("title", "subtitle", "intro", "text", "description", "label", "cta"):
            v = ce.get(k)
            if isinstance(v, str) and v.strip():
                parts.append(v)
        md = ce.get("metadata")
        if isinstance(md, dict):
            for k in ("title", "subtitle", "description", "footnote"):
                v = md.get(k)
                if isinstance(v, str) and v.strip():
                    parts.append(v)

    return " ".join(parts)

def segment_text_hash(seg: Dict[str, Any], mode: str = "head", n_chars: int = 180) -> str:
    txt = normalize_ws(strip_html(extract_text_from_segment(seg)))
    if not txt:
        return ""
    snippet = txt[:n_chars] if mode == "head" else txt[-n_chars:]
    return sha256(snippet)

def template_id(seg: Dict[str, Any]) -> str:
    return str(seg.get("template_id") or "")

def load_registry(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        save_json([], path)
        print(f"🆕 Created empty poll registry: {path}")
        return []
    data = load_json(path)
    if data is None:
        return []
    if isinstance(data, dict):
        return list(data.values())
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected registry format: {type(data)}")

def poll_already_present(segments: List[dict], poll_id: str) -> bool:
    for seg in segments:
        if isinstance(seg, dict) and seg.get("template_id") == "poll":
            c = seg.get("content", {})
            if isinstance(c, dict) and c.get("pollId") == poll_id:
                return True
    return False

def anchor_score_pair(before_seg, after_seg, before_anchor, after_anchor) -> int:
    score = 0
    if template_id(before_seg) == (before_anchor.get("template_id") or ""):
        score += 10
    if template_id(after_seg) == (after_anchor.get("template_id") or ""):
        score += 10

    b_hash = before_anchor.get("text_hash")
    a_hash = after_anchor.get("text_hash")

    if b_hash:
        if segment_text_hash(before_seg, "head") == b_hash or segment_text_hash(before_seg, "tail") == b_hash:
            score += 25
        else:
            score -= 5
    if a_hash:
        if segment_text_hash(after_seg, "head") == a_hash or segment_text_hash(after_seg, "tail") == a_hash:
            score += 25
        else:
            score -= 5

    return score

def find_best_insertion_index(segments, before_anchor, after_anchor) -> Optional[int]:
    best = None
    best_score = -10_000

    # Adjacent pair scan
    for i in range(len(segments) - 1):
        b = segments[i]
        a = segments[i+1]
        if not isinstance(b, dict) or not isinstance(a, dict):
            continue
        score = anchor_score_pair(b, a, before_anchor, after_anchor)
        if score > best_score:
            best_score = score
            best = i+1

    if best is not None and best_score >= 20:
        return best

    # One-sided fallback
    after_tid = after_anchor.get("template_id") or ""
    before_tid = before_anchor.get("template_id") or ""

    if after_tid:
        for i, seg in enumerate(segments):
            if isinstance(seg, dict) and template_id(seg) == after_tid:
                return i

    if before_tid:
        for i in range(len(segments)-1, -1, -1):
            seg = segments[i]
            if isinstance(seg, dict) and template_id(seg) == before_tid:
                return i+1

    # Heuristic fallback
    for i, seg in enumerate(segments):
        if isinstance(seg, dict) and template_id(seg) == "key_concepts":
            return i+1
    for i, seg in enumerate(segments):
        if isinstance(seg, dict) and template_id(seg) == "text":
            return i+1

    return None

def insert_poll_segment(segments: List[dict], insert_at: int, poll_segment: dict) -> List[dict]:
    insert_at = max(0, min(insert_at, len(segments)))
    return segments[:insert_at] + [poll_segment] + segments[insert_at:]

def validate_poll_segment(seg: Dict[str, Any]) -> Tuple[bool, str]:
    if not isinstance(seg, dict):
        return False, "segment not dict"
    if seg.get("template_id") != "poll":
        return False, "template_id not poll"
    c = seg.get("content", {})
    if not isinstance(c, dict) or not c.get("pollId"):
        return False, "missing content.pollId"
    opts = c.get("options", [])
    if not isinstance(opts, list) or len(opts) < 2:
        return False, "options missing/too few"
    return True, ""

def run_insert_polls():
    registry = load_registry(POLL_REGISTRY_PATH)
    if not registry:
        print("✅ Poll registry is empty. Nothing to insert.")
        return

    inserted = 0
    skipped_exists = 0
    skipped_missing_file = 0
    failed = 0

    for rec in registry:
        poll_id = rec.get("pollId") or rec.get("segment", {}).get("content", {}).get("pollId")
        lesson_id = rec.get("lesson_id") or rec.get("lessonId")
        poll_segment = rec.get("segment")

        if not poll_id or not lesson_id or not poll_segment:
            print(f"⚠️ Invalid registry record, skipping: {rec.get('pollId')}")
            failed += 1
            continue

        ok, msg = validate_poll_segment(poll_segment)
        if not ok:
            print(f"⚠️ Invalid poll segment {poll_id}: {msg}")
            failed += 1
            continue

        path = lesson_path_en(lesson_id)
        if not path.exists():
            print(f"⚠️ Lesson missing for poll {poll_id}: {path}")
            skipped_missing_file += 1
            continue

        lesson = load_json(path)
        if not lesson or not isinstance(lesson.get("segments"), list):
            print(f"⚠️ Bad lesson JSON: {path}")
            failed += 1
            continue

        segments = lesson["segments"]
        if poll_already_present(segments, poll_id):
            skipped_exists += 1
            continue

        ins = rec.get("insert", {}) if isinstance(rec.get("insert", {}), dict) else {}
        before_anchor = ins.get("before", {}) if isinstance(ins.get("before", {}), dict) else {}
        after_anchor  = ins.get("after", {}) if isinstance(ins.get("after", {}), dict) else {}

        idx = find_best_insertion_index(segments, before_anchor, after_anchor)
        if idx is None:
            print(f"⚠️ Could not place poll {poll_id} in {path}")
            failed += 1
            continue

        lesson["segments"] = insert_poll_segment(segments, idx, poll_segment)
        save_json(lesson, path)
        inserted += 1
        print(f"✅ Inserted poll {poll_id} into {path} at index {idx}")

    print("\n📌 Poll Insertion Summary")
    print(f"✅ Inserted: {inserted}")
    print(f"⏭️ Skipped (already exists): {skipped_exists}")
    print(f"⚠️ Skipped (missing lesson file): {skipped_missing_file}")
    print(f"❌ Failed: {failed}")

run_insert_polls()


✅ Inserted poll poll-1.1.0-list_of_lessons_to_connection_next-a8f89e into ../03_Outputs/SEA_Modules/en/Module_1/1.1.0.json at index 4
✅ Inserted poll poll-1.1.1-echarts_chart_to_text-e9099f into ../03_Outputs/SEA_Modules/en/Module_1/1.1.1.json at index 10
✅ Inserted poll poll-1.1.2-echarts_chart_to_text-530691 into ../03_Outputs/SEA_Modules/en/Module_1/1.1.2.json at index 13
✅ Inserted poll poll-1.1.3-echarts_chart_to_text-85c2a2 into ../03_Outputs/SEA_Modules/en/Module_1/1.1.3.json at index 13
✅ Inserted poll poll-1.2.0-list_of_lessons_to_connection_next-85939c into ../03_Outputs/SEA_Modules/en/Module_1/1.2.0.json at index 3
✅ Inserted poll poll-1.2.2-key_concepts_to_text-f91d92 into ../03_Outputs/SEA_Modules/en/Module_1/1.2.2.json at index 3
✅ Inserted poll poll-1.2.3-key_concepts_to_text-717488 into ../03_Outputs/SEA_Modules/en/Module_1/1.2.3.json at index 3
✅ Inserted poll poll-1.3.0-list_of_lessons_to_connection_next-86228f into ../03_Outputs/SEA_Modules/en/Module_1/1.3.0.json at 